# Entrega 3 - Classificação de Intenções

**Curso:** Processamento de Linguagem Natural  
**Aluno:** gisele
**Data:** 30 de jan

---

## Contexto e objetivo

Você irá implementar o módulo de **classificação de intenções** do chatbot de e-commerce:

1. **Preparação de dados** para treinar classificadores de texto usando TF-IDF.
2. **Treinamento de classificadores** com Logistic Regression e otimização de hiperparâmetros.
3. **Classificação de intenções** com níveis de confiança e threshold.

O que você implementar aqui **será reutilizado cumulativamente** nas próximas entregas e no Projeto Final. Portanto, trate cada função como parte de uma API estável: assinaturas, nomes e tipos esperados devem ser preservados.

---

## Instruções

1. Complete todos os exercícios marcados com `# === SEU CÓDIGO AQUI ===`
2. **Não modifique** os nomes das funções - eles serão usados no Projeto Final
3. Execute todas as células em ordem antes de enviar
4. **Testes ocultos** serão executados na correção automática

---

## Critérios de Avaliação

| Questão | Pontos | Descrição |
|---------|--------|------------|
| Q1 | 10 | Texto para vetor TF-IDF |
| Q2 | 15 | Preparar dados para classificação |
| Q3 | 15 | Treinar classificador |
| Q4 | 10 | Otimizar hiperparâmetros |
| Q5 | 15 | Classificar intenção (**CRÍTICO**) |
| Q6 | 12 | Classificar com threshold |
| Q7 | 10 | Classificar em lote |
| Q8 | 13 | Avaliar classificador |
| **Total** | **100** | |

---

## Conteúdo Avaliado

- **Aula 5:** Word Embeddings e representações vetoriais
- **Aula 6:** Classificação de Textos e Análise de Sentimentos

---

## Funções a Implementar

```python
# Preparação de Dados
texto_para_vetor_tfidf(texto, vetorizador) -> np.ndarray
preparar_dados_classificacao(textos, labels) -> tuple

# Treinamento
treinar_classificador(X_train, y_train) -> LogisticRegression
otimizar_hiperparametros(X_train, y_train) -> dict

# Classificação
classificar_intencao(texto, classificador, vetorizador) -> tuple
classificar_com_threshold(texto, classificador, vetorizador, threshold) -> dict
classificar_lote(textos, classificador, vetorizador) -> list

# Avaliação
avaliar_classificador(classificador, X_test, y_test) -> dict
```

Notas:
- A implementação é avaliada pelo **comportamento observável**: saídas, tipos e chaves retornadas.
- Este notebook impõe técnicas específicas por design, pois o resultado será consumido por etapas posteriores.

## Setup

Esta seção prepara o ambiente e carrega dependências e dados do problema.

Requisitos:
- Execute as células desta seção **sem alterações**, exceto quando o notebook indicar explicitamente o contrário.
- Se houver erro de importação/instalação, corrija o ambiente (não altere a assinatura das funções avaliadas em nenhuma hipótese).

In [1]:
# === SETUP ===
!pip install nltk scikit-learn pandas --quiet

import numpy as np
import pandas as pd
import random
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# === CONFIGURAÇÃO DE VALIDAÇÃO (NÃO MODIFICAR) ===
SEED_AVALIACAO = 42
np.random.seed(SEED_AVALIACAO)
random.seed(SEED_AVALIACAO)

# Parâmetros obrigatórios de split
SPLIT_TEST_SIZE = 0.2
SPLIT_RANDOM_STATE = 42
SPLIT_TRAIN_SIZE = 1.0 - SPLIT_TEST_SIZE

# Parâmetros obrigatórios do vetorizador
TFIDF_MAX_FEATURES = 500
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 1

# Índices específicos de validação - SERÃO USADOS NOS TESTES
VALIDACAO_SAUDACAO_IDX = 3
VALIDACAO_RASTREAMENTO_IDX = 25
VALIDACAO_DEVOLUCAO_IDX = 45
VALIDACAO_RECLAMACAO_IDX = 65

# Sanity checks
assert 0 < SPLIT_TEST_SIZE < 1, "SPLIT_TEST_SIZE deve estar em (0,1)"
assert TFIDF_MAX_FEATURES >= 10, "TFIDF_MAX_FEATURES muito baixo"
assert isinstance(TFIDF_NGRAM_RANGE, tuple) and len(TFIDF_NGRAM_RANGE) == 2
assert TFIDF_NGRAM_RANGE[0] <= TFIDF_NGRAM_RANGE[1]

print("==> Setup completo!")

==> Setup completo!


In [2]:
# === CARREGAR DATASET DE INTENÇÕES ===
# Não modifique esta célula

INTENCOES_TEXTOS = [
    # SAUDACAO (20)
    "oi", "ola", "bom dia", "boa tarde", "boa noite",
    "oi tudo bem", "ola como vai", "hey", "opa", "e ai",
    "oi boa tarde", "ola bom dia", "oie", "oii", "olaa",
    "oi pode me ajudar", "ola preciso de ajuda", "oi to precisando de ajuda",
    "bom dia preciso de ajuda", "boa noite podem me ajudar",

    # RASTREAMENTO (20)
    "onde esta meu pedido", "quero rastrear meu pedido", "cade minha encomenda",
    "meu pedido ja foi enviado", "quando meu pedido vai chegar",
    "status do pedido", "rastrear pedido", "rastreamento",
    "onde esta o pedido PED-123456", "rastrear PED-789",
    "minha encomenda esta onde", "quero saber do meu pedido",
    "acompanhar entrega", "ver status da entrega", "entrega do pedido",
    "pedido nao chegou ainda", "ta demorando meu pedido",
    "quando chega meu pedido", "previsao de entrega", "prazo do pedido",

    # DEVOLUCAO (20)
    "quero devolver", "como devolver produto", "devolucao",
    "quero trocar o produto", "nao quero mais o produto",
    "como faco para devolver", "politica de devolucao", "prazo para devolver",
    "posso devolver", "quero fazer troca", "trocar por outro",
    "devolucao de produto", "devolver compra", "cancelar e devolver",
    "como funciona devolucao", "regras de troca", "quero meu dinheiro",
    "reembolso do produto", "estornar compra", "devolver pedido",

    # RECLAMACAO (20)
    "produto veio errado", "recebi produto errado", "veio com defeito",
    "produto danificado", "nao era isso que pedi", "muito insatisfeito",
    "pessimo atendimento", "vou processar", "reclamar",
    "produto quebrado", "veio faltando peca", "qualidade horrivel",
    "quero reclamar", "fazer reclamacao", "registrar queixa",
    "absurdo", "vergonha", "nunca mais compro",
    "vou no procon", "vou no reclame aqui",

    # PAGAMENTO (20)
    "formas de pagamento", "como pagar", "aceita cartao",
    "posso pagar com pix", "tem boleto", "parcela em quantas vezes",
    "pagar com credito", "pagar com debito", "opcoes de pagamento",
    "como parcelar", "juros no cartao", "pix tem desconto",
    "boleto tem desconto", "pagar a vista", "meios de pagamento",
    "cartao de credito", "cartao de debito", "pagamento por pix",
    "gerar boleto", "segunda via boleto",

    # FAQ (20)
    "como funciona", "duvida", "pergunta",
    "quero saber", "informacao", "me explica",
    "como faz", "qual o procedimento", "como proceder",
    "tenho uma duvida", "pode me explicar", "nao entendi",
    "como usar cupom", "aplicar desconto", "codigo promocional",
    "frete gratis", "valor do frete", "calcular frete",
    "horario de funcionamento", "telefone de contato",

    # DESPEDIDA (20)
    "tchau", "ate mais", "obrigado", "valeu", "brigado",
    "muito obrigado", "agradeco", "era so isso", "so isso",
    "tchau obrigado", "ate logo", "falou", "flw",
    "obrigado pela ajuda", "valeu pela ajuda", "resolvido",
    "era isso", "ja resolvi", "consegui resolver", "tudo certo"
]

INTENCOES_LABELS = (
    ['saudacao'] * 20 +
    ['rastreamento'] * 20 +
    ['devolucao'] * 20 +
    ['reclamacao'] * 20 +
    ['pagamento'] * 20 +
    ['faq'] * 20 +
    ['despedida'] * 20
)

# Validação de índices específicos
TEXTO_VALIDACAO_SAUDACAO = INTENCOES_TEXTOS[VALIDACAO_SAUDACAO_IDX]
TEXTO_VALIDACAO_RASTREAMENTO = INTENCOES_TEXTOS[VALIDACAO_RASTREAMENTO_IDX]
TEXTO_VALIDACAO_DEVOLUCAO = INTENCOES_TEXTOS[VALIDACAO_DEVOLUCAO_IDX]
TEXTO_VALIDACAO_RECLAMACAO = INTENCOES_TEXTOS[VALIDACAO_RECLAMACAO_IDX]

print(f"Dataset: {len(INTENCOES_TEXTOS)} exemplos")
print(f"Classes: {sorted(set(INTENCOES_LABELS))}")
print(f"Distribuição: {Counter(INTENCOES_LABELS)}")
print(f"\nTextos de validação:")
print(f"  - Saudação [{VALIDACAO_SAUDACAO_IDX}]: '{TEXTO_VALIDACAO_SAUDACAO}'")
print(f"  - Rastreamento [{VALIDACAO_RASTREAMENTO_IDX}]: '{TEXTO_VALIDACAO_RASTREAMENTO}'")

Dataset: 140 exemplos
Classes: ['despedida', 'devolucao', 'faq', 'pagamento', 'rastreamento', 'reclamacao', 'saudacao']
Distribuição: Counter({'saudacao': 20, 'rastreamento': 20, 'devolucao': 20, 'reclamacao': 20, 'pagamento': 20, 'faq': 20, 'despedida': 20})

Textos de validação:
  - Saudação [3]: 'boa tarde'
  - Rastreamento [25]: 'status do pedido'


---

## Parte 1: Preparação de Dados (25 pontos)

Nesta seção você implementará funções para transformar textos em vetores TF-IDF e preparar os dados para treinamento do classificador.

### Questão 1: Texto para Vetor TF-IDF (10 pontos)

### Objetivo
Converter um texto em um vetor TF-IDF denso (1D) usando um vetorizador já treinado.

### Entradas esperadas
- `texto: str` contendo o texto a ser vetorizado.
- `vetorizador: TfidfVectorizer` já treinado (fitted) em um corpus.

### Processamento obrigatório
Implemente `texto_para_vetor_tfidf(texto, vetorizador)` de forma que:

1. Use o método `transform()` do vetorizador para converter o texto em matriz esparsa.
   - Lembre-se que `transform()` espera uma lista, mesmo para um único texto.
2. Converta a matriz esparsa para array denso usando `toarray()`.
3. Extraia o vetor 1D (primeira linha da matriz).
4. Retorne o vetor como `np.ndarray` unidimensional.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `texto_para_vetor_tfidf(texto: str, vetorizador: TfidfVectorizer) -> np.ndarray`
- O array retornado deve ser 1D com shape `(n_features,)`
- O tipo deve ser `numpy.ndarray`, não matriz esparsa

### Restrições técnicas
- O vetorizador passado já está treinado; não chame `fit()` novamente.
- O retorno deve ser exatamente 1D (use indexação `[0]` após `toarray()`).
- Não normalize o vetor além do que o TF-IDF já faz.

### Observações de continuidade
Esta função é usada internamente por `classificar_intencao()` e outras funções desta entrega. No Projeto Final, ela será adaptada para suportar múltiplos vetorizadores.

In [3]:
def texto_para_vetor_tfidf(texto: str, vetorizador: TfidfVectorizer) -> np.ndarray:
    """
    Converte texto em vetor TF-IDF denso.

    Args:
        texto: Texto a ser vetorizado
        vetorizador: TfidfVectorizer já treinado

    Returns:
        np.ndarray 1D com representação TF-IDF
    """
    # === SEU CÓDIGO AQUI ===
    vetor = vetorizador.transform([texto])
    return vetor.toarray()[0]

In [4]:
# Teste Q1
vetorizador_teste_q1 = TfidfVectorizer(
    ngram_range=TFIDF_NGRAM_RANGE,
    max_features=TFIDF_MAX_FEATURES
)
vetorizador_teste_q1.fit(INTENCOES_TEXTOS)

vetor_q1 = texto_para_vetor_tfidf("quero rastrear meu pedido", vetorizador_teste_q1)

print(f"Q1: Vetor shape: {vetor_q1.shape}")
print(f"    Tipo: {type(vetor_q1)}")
print(f"    Soma: {vetor_q1.sum():.4f}")
print(f"    Valores não-zero: {np.count_nonzero(vetor_q1)}")

# Validações obrigatórias
assert isinstance(vetor_q1, np.ndarray), "Retorno deve ser np.ndarray"
assert len(vetor_q1.shape) == 1, "Vetor deve ser 1D"
assert vetor_q1.shape[0] <= TFIDF_MAX_FEATURES, f"Shape deve ser menor ou igual a ({TFIDF_MAX_FEATURES},)"
print("\n✓ Q1 OK!")

Q1: Vetor shape: (360,)
    Tipo: <class 'numpy.ndarray'>
    Soma: 2.6055
    Valores não-zero: 7

✓ Q1 OK!


### Questão 2: Preparar Dados para Classificação (15 pontos)

### Objetivo
Criar pipeline completo de preparação: treinar vetorizador TF-IDF e dividir dados em treino/teste.

### Entradas esperadas
- `textos: list[str]` lista de textos de entrada.
- `labels: list[str]` lista de labels correspondentes.

### Processamento obrigatório
Implemente `preparar_dados_classificacao(textos, labels)` de forma que:

1. Crie um `TfidfVectorizer` com os parâmetros **obrigatórios**:
   - `ngram_range=TFIDF_NGRAM_RANGE` (definido como `(1, 2)`)
   - `max_features=TFIDF_MAX_FEATURES` (definido como `500`)
   - `min_df=TFIDF_MIN_DF` (definido como `1`)
   - `lowercase=True`
2. Treine (`fit`) o vetorizador nos textos.
3. Transforme todos os textos em matriz TF-IDF.
4. Divida os dados usando `train_test_split` com parâmetros **obrigatórios**:
   - `test_size=SPLIT_TEST_SIZE` (definido como `0.2`)
   - `random_state=SPLIT_RANDOM_STATE` (definido como `42`)
   - `stratify=labels` para manter proporção das classes (use a função `np.array(labels)` para converter os labels para um formato adequado)   
5. Retorne tupla com 5 elementos na ordem especificada.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `preparar_dados_classificacao(textos: list, labels: list) -> tuple`
- A tupla retornada deve conter exatamente: `(X_train, X_test, y_train, y_test, vetorizador)`
- `X_train` e `X_test` devem ser matrizes esparsas ou arrays
- `vetorizador` deve ser o `TfidfVectorizer` já treinado

### Restrições técnicas
- Use **exatamente** os parâmetros especificados nas constantes.
- O split deve usar `stratify` para garantir distribuição balanceada.
- A ordem dos elementos na tupla de retorno é obrigatória.
- O vetorizador retornado deve estar fitted (pronto para `transform()`).

### Observações de continuidade
Esta função é o ponto de entrada para preparação de dados. O vetorizador retornado será usado em todas as funções de classificação subsequentes e no Projeto Final.

In [5]:
def preparar_dados_classificacao(textos: list, labels: list) -> tuple:
    """
    Prepara dados para treinamento de classificador.

    Args:
        textos: Lista de textos
        labels: Lista de labels correspondentes

    Returns:
        tuple: (X_train, X_test, y_train, y_test, vetorizador)
    """
    # === SEU CÓDIGO AQUI ===
    vetorizador = TfidfVectorizer(
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        lowercase=True
    )

    X = vetorizador.fit_transform(textos)
    y = np.array(labels)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=SPLIT_TEST_SIZE,
        random_state=SPLIT_RANDOM_STATE,
        stratify=y
    )

    return X_train, X_test, y_train, y_test, vetorizador

In [6]:
# Teste Q2
X_train, X_test, y_train, y_test, vetorizador = preparar_dados_classificacao(
    INTENCOES_TEXTOS, INTENCOES_LABELS
)

print(f"Q2: Dados preparados")
print(f"    X_train shape: {X_train.shape}")
print(f"    X_test shape: {X_test.shape}")
print(f"    y_train: {len(y_train)} exemplos")
print(f"    y_test: {len(y_test)} exemplos")
print(f"    Vocabulário: {len(vetorizador.vocabulary_)} termos")

# --------------------------
# Validações obrigatórias
# --------------------------

# Tipos básicos
assert isinstance(vetorizador, TfidfVectorizer), "vetorizador deve ser TfidfVectorizer"
assert hasattr(vetorizador, "vocabulary_"), "vetorizador deve estar fitted"
assert isinstance(y_train, np.ndarray) and isinstance(y_test, np.ndarray), "y_train/y_test devem ser np.ndarray"

# Contagens (robusto a arredondamento)
total_exemplos = len(INTENCOES_TEXTOS)
assert X_train.shape[0] + X_test.shape[0] == total_exemplos, "Split deve preservar o total de exemplos"
assert y_train.shape[0] == X_train.shape[0], "y_train deve alinhar com X_train"
assert y_test.shape[0] == X_test.shape[0], "y_test deve alinhar com X_test"

# Proporção aproximada do split (tolerância de 1 exemplo)
expected_test = int(round(total_exemplos * SPLIT_TEST_SIZE))
assert abs(X_test.shape[0] - expected_test) <= 1, f"X_test deve ter ~{expected_test} exemplos"
assert abs(X_train.shape[0] - (total_exemplos - expected_test)) <= 1, "X_train deve ter o complementar"

# Dimensionalidade: deve bater com o vocabulário treinado
n_features = len(vetorizador.vocabulary_)
assert X_train.shape[1] == n_features and X_test.shape[1] == n_features, "Features devem bater com vocabulary_"
assert n_features <= TFIDF_MAX_FEATURES, f"n_features deve ser <= TFIDF_MAX_FEATURES ({TFIDF_MAX_FEATURES})"

# Checar uso das constantes (anti-hardcode)
params = vetorizador.get_params()
assert params["ngram_range"] == TFIDF_NGRAM_RANGE, "ngram_range deve usar TFIDF_NGRAM_RANGE"
assert params["max_features"] == TFIDF_MAX_FEATURES, "max_features deve usar TFIDF_MAX_FEATURES"
assert params["min_df"] == TFIDF_MIN_DF, "min_df deve usar TFIDF_MIN_DF"
assert params["lowercase"] is True, "lowercase deve ser True"

# Verificar stratify (efeito): todas as classes presentes e distribuição razoável
train_dist = Counter(y_train.tolist())
test_dist = Counter(y_test.tolist())
all_dist = Counter(INTENCOES_LABELS)

assert set(train_dist.keys()) == set(all_dist.keys()), "Todas as classes devem aparecer no treino (stratify)"
assert set(test_dist.keys()) == set(all_dist.keys()), "Todas as classes devem aparecer no teste (stratify)"

print(f"\n    Distribuição total:  {dict(all_dist)}")
print(f"    Distribuição treino: {dict(train_dist)}")
print(f"    Distribuição teste:  {dict(test_dist)}")

print("\n✓ Q2 OK!")

Q2: Dados preparados
    X_train shape: (112, 360)
    X_test shape: (28, 360)
    y_train: 112 exemplos
    y_test: 28 exemplos
    Vocabulário: 360 termos

    Distribuição total:  {'saudacao': 20, 'rastreamento': 20, 'devolucao': 20, 'reclamacao': 20, 'pagamento': 20, 'faq': 20, 'despedida': 20}
    Distribuição treino: {'devolucao': 16, 'despedida': 16, 'reclamacao': 16, 'saudacao': 16, 'faq': 16, 'pagamento': 16, 'rastreamento': 16}
    Distribuição teste:  {'pagamento': 4, 'rastreamento': 4, 'devolucao': 4, 'reclamacao': 4, 'despedida': 4, 'saudacao': 4, 'faq': 4}

✓ Q2 OK!


---

## Parte 2: Treinamento de Classificadores (25 pontos)

Nesta seção você implementará funções para treinar e otimizar classificadores de texto.

### Questão 3: Treinar Classificador (15 pontos)

### Objetivo
Treinar um classificador Logistic Regression com configuração padrão otimizada para classificação multiclasse de textos.

### Entradas esperadas
- `X_train` matriz de features TF-IDF (esparsa ou densa).
- `y_train` lista ou array de labels.

### Processamento obrigatório
Implemente `treinar_classificador(X_train, y_train)` de forma que:

1. Crie um `LogisticRegression` com os parâmetros **obrigatórios**:
   - `max_iter=1000` para garantir convergência
   - `random_state=SEED_AVALIACAO` (definido como `42`)
   - `multi_class='multinomial'` para classificação multiclasse
   - `solver='lbfgs'` compatível com multinomial
   - `C=1.0` regularização padrão
2. Treine (`fit`) o classificador nos dados.
3. Retorne o classificador treinado.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `treinar_classificador(X_train, y_train) -> LogisticRegression`
- O classificador retornado deve estar treinado (ter atributo `classes_`)
- Deve suportar `predict()` e `predict_proba()`

### Restrições técnicas
- Use **exatamente** os parâmetros especificados.
- Não adicione parâmetros extras ao classificador.
- O classificador deve estar pronto para predição imediatamente após retorno.

### Observações de continuidade
Este classificador é o componente central do sistema de intenções. No Projeto Final, ele será integrado com threshold dinâmico e fallback para atendimento humano.

In [7]:
def treinar_classificador(X_train, y_train) -> LogisticRegression:
    """
    Treina classificador Logistic Regression.

    Args:
        X_train: Matriz de features de treino
        y_train: Labels de treino

    Returns:
        LogisticRegression treinado
    """
    # === SEU CÓDIGO AQUI ===
    classificador = LogisticRegression(
        max_iter=1000,
        random_state=SEED_AVALIACAO,
        multi_class='multinomial',
        solver='lbfgs',
        C=1.0
    )

    classificador.fit(X_train, y_train)
    return classificador

In [8]:
# Teste Q3
classificador = treinar_classificador(X_train, y_train)

print(f"Q3: Classificador treinado")
print(f"    Tipo: {type(classificador).__name__}")
print(f"    Classes: {list(classificador.classes_)}")
print(f"    Coeficientes shape: {classificador.coef_.shape}")

# Validações obrigatórias
assert isinstance(classificador, LogisticRegression), "Deve ser LogisticRegression"
assert hasattr(classificador, 'classes_'), "Classificador deve estar treinado"
assert len(classificador.classes_) == len(set(y_train)), "Deve ter 7 classes"
assert hasattr(classificador, 'predict_proba'), "Deve suportar predict_proba"

# Verificar parâmetros obrigatórios
assert classificador.max_iter == 1000, "max_iter deve ser 1000"
assert classificador.random_state == SEED_AVALIACAO, f"random_state deve ser {SEED_AVALIACAO}"
assert classificador.multi_class == 'multinomial', "multi_class deve ser 'multinomial'"

# Teste rápido de predição
pred_teste = classificador.predict(X_test[:5])
print(f"\n    Predições teste: {list(pred_teste)}")

print("\n✓ Q3 OK!")

Q3: Classificador treinado
    Tipo: LogisticRegression
    Classes: [np.str_('despedida'), np.str_('devolucao'), np.str_('faq'), np.str_('pagamento'), np.str_('rastreamento'), np.str_('reclamacao'), np.str_('saudacao')]
    Coeficientes shape: (7, 360)

    Predições teste: [np.str_('pagamento'), np.str_('rastreamento'), np.str_('rastreamento'), np.str_('pagamento'), np.str_('reclamacao')]

✓ Q3 OK!


### Questão 4: Otimizar Hiperparâmetros (10 pontos)

### Objetivo
Encontrar os melhores hiperparâmetros para o classificador usando Grid Search com validação cruzada.

### Entradas esperadas
- `X_train` matriz de features TF-IDF.
- `y_train` lista ou array de labels.

### Processamento obrigatório
Implemente `otimizar_hiperparametros(X_train, y_train)` de forma que:

1. Defina grade de parâmetros **obrigatória**:
   ```python
   param_grid = {
       'C': [0.1, 1.0, 10.0],
       'solver': ['lbfgs', 'saga']
   }
   ```
2. Crie `LogisticRegression` base com:
   - `max_iter=1000`
   - `random_state=SEED_AVALIACAO`
   - `multi_class='multinomial'`
3. Execute `GridSearchCV` com:
   - `cv=3` (3-fold cross-validation)
   - `scoring='f1_macro'`
4. Retorne dicionário com resultados.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `otimizar_hiperparametros(X_train, y_train) -> dict`
- O dicionário retornado deve conter **exatamente** as chaves:
  - `'best_params'`: dict com melhores parâmetros
  - `'best_score'`: float com melhor score
  - `'best_estimator'`: o classificador otimizado

### Restrições técnicas
- Use **exatamente** a grade de parâmetros especificada.
- Use `cv=3` e `scoring='f1_macro'`.
- Retorne todas as chaves obrigatórias.

### Observações de continuidade
Esta função permite encontrar a melhor configuração do classificador. No Projeto Final, os parâmetros otimizados podem ser usados para inicialização.

In [9]:
def otimizar_hiperparametros(X_train, y_train) -> dict:
    """
    Otimiza hiperparâmetros do classificador via GridSearchCV.

    Args:
        X_train: Matriz de features de treino
        y_train: Labels de treino

    Returns:
        dict com 'best_params', 'best_score', 'best_estimator'
    """
    # === SEU CÓDIGO AQUI ===
    param_grid = {
        'C': [0.1, 1.0, 10.0],
        'solver': ['lbfgs', 'saga']
    }

    base_model = LogisticRegression(
        max_iter=1000,
        random_state=SEED_AVALIACAO,
        multi_class='multinomial'
    )

    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=3,
        scoring='f1_macro'
    )

    grid.fit(X_train, y_train)

    return {
        'best_params': grid.best_params_,
        'best_score': float(grid.best_score_),
        'best_estimator': grid.best_estimator_
    }

In [10]:
# Teste Q4
resultado_otimizacao = otimizar_hiperparametros(X_train, y_train)

print(f"Q4: Otimização concluída")
print(f"    Melhores parâmetros: {resultado_otimizacao['best_params']}")
print(f"    Melhor score (F1-macro): {resultado_otimizacao['best_score']:.4f}")
print(f"    Estimador: {type(resultado_otimizacao['best_estimator']).__name__}")

# Validações obrigatórias
assert isinstance(resultado_otimizacao, dict), "Retorno deve ser dict"
assert set(resultado_otimizacao.keys()) == {'best_params', 'best_score', 'best_estimator'}, "Dicionário de retorno deve conter exatamente as chaves obrigatórias especificadas no enunciado"
assert 'best_params' in resultado_otimizacao, "Deve conter 'best_params'"
assert 'best_score' in resultado_otimizacao, "Deve conter 'best_score'"
assert 'best_estimator' in resultado_otimizacao, "Deve conter 'best_estimator'"
assert 'C' in resultado_otimizacao['best_params'], "best_params deve conter 'C'"
assert 'solver' in resultado_otimizacao['best_params'], "best_params deve conter 'solver'"

print("\n✓ Q4 OK!")

Q4: Otimização concluída
    Melhores parâmetros: {'C': 10.0, 'solver': 'lbfgs'}
    Melhor score (F1-macro): 0.6352
    Estimador: LogisticRegression

✓ Q4 OK!


---

## Parte 3: Classificação e Avaliação (50 pontos)

Nesta seção você implementará as funções de classificação que serão usadas diretamente no chatbot e funções de avaliação de desempenho.

### Questão 5: Classificar Intenção (15 pontos) - **CRÍTICO**

### Objetivo
Classificar a intenção de um texto, retornando a classe predita e o nível de confiança (probabilidade).

### Entradas esperadas
- `texto: str` mensagem do usuário a classificar.
- `classificador: LogisticRegression` já treinado.
- `vetorizador: TfidfVectorizer` já treinado.

### Processamento obrigatório
Implemente `classificar_intencao(texto, classificador, vetorizador)` de forma que:

1. Vetorize o texto usando o vetorizador (método `transform`).
   - Lembre-se que `transform` espera lista.
2. Obtenha a predição usando `predict()`.
3. Obtenha as probabilidades usando `predict_proba()`.
4. Identifique o índice da classe predita em `classificador.classes_`.
5. Extraia a probabilidade correspondente à classe predita.
6. Retorne tupla `(intenção, confiança)` com tipos corretos.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `classificar_intencao(texto: str, classificador, vetorizador) -> tuple`
- A tupla retornada deve conter: `(intenção: str, confiança: float)`
- `intenção` deve ser uma das 7 classes do dataset
- `confiança` deve estar no intervalo `[0.0, 1.0]`

### Restrições técnicas
- Retorne `str` para intenção (não numpy.str_).
- Retorne `float` nativo para confiança (não numpy.float64).
- A confiança deve ser a probabilidade da classe predita (retornada por `predict()`, usando o índice correspondente em classificador.classes_), e NÃO o máximo de todas.

### Observações de continuidade
**Esta é a função principal do módulo de classificação.** Ela será usada diretamente no Projeto Final para rotear mensagens. No Projeto Final, você expandirá esta função para incluir threshold dinâmico e flag de escalação para humano.

In [11]:
def classificar_intencao(texto: str, classificador, vetorizador) -> tuple:
    """
    Classifica a intenção de um texto.

    Args:
        texto: Mensagem do usuário
        classificador: LogisticRegression treinado
        vetorizador: TfidfVectorizer treinado

    Returns:
        tuple: (intenção: str, confiança: float)
    """
    # === SEU CÓDIGO AQUI ===
    vetor = vetorizador.transform([texto])

    pred = classificador.predict(vetor)[0]
    probas = classificador.predict_proba(vetor)[0]

    classes = list(classificador.classes_)
    idx = classes.index(pred)

    confianca = float(probas[idx])

    return str(pred), confianca

In [12]:
# Teste Q5
testes_q5 = [
    ("oi bom dia", "saudacao"),
    ("onde esta meu pedido", "rastreamento"),
    ("quero devolver", "devolucao"),
    ("produto veio quebrado", "reclamacao"),
    ("aceita cartao", "pagamento"),
    ("tchau obrigado", "despedida"),
]

print("Q5: Testes de classificação:")
acertos = 0
for texto, esperado in testes_q5:
    intencao, conf = classificar_intencao(texto, classificador, vetorizador)
    status = "✓" if intencao == esperado else "✗"
    if intencao == esperado:
        acertos += 1
    print(f"    {status} '{texto}' -> {intencao} ({conf:.2%}) [esperado: {esperado}]")

print(f"\n    Acurácia nos testes: {acertos}/{len(testes_q5)}")

# Confiança deve bater com a probabilidade da classe predita
vet = vetorizador.transform([TEXTO_VALIDACAO_SAUDACAO])
pred = classificador.predict(vet)[0]
proba = classificador.predict_proba(vet)[0]
idx = list(classificador.classes_).index(pred)
expected_conf = float(proba[idx])

int_val, conf_val = classificar_intencao(TEXTO_VALIDACAO_SAUDACAO, classificador, vetorizador)
assert abs(conf_val - expected_conf) < 1e-12, "Confiança deve corresponder à probabilidade da classe predita"


# Validações obrigatórias com índices específicos
int_val, conf_val = classificar_intencao(TEXTO_VALIDACAO_SAUDACAO, classificador, vetorizador)
assert isinstance(int_val, str), "Intenção deve ser str"
assert isinstance(conf_val, float), "Confiança deve ser float"
assert 0.0 <= conf_val <= 1.0, "Confiança deve estar em [0, 1]"
assert int_val in classificador.classes_, "Intenção deve ser classe válida"

print("\n✓ Q5 OK!")

Q5: Testes de classificação:
    ✓ 'oi bom dia' -> saudacao (50.98%) [esperado: saudacao]
    ✓ 'onde esta meu pedido' -> rastreamento (54.20%) [esperado: rastreamento]
    ✓ 'quero devolver' -> devolucao (38.99%) [esperado: devolucao]
    ✓ 'produto veio quebrado' -> reclamacao (31.11%) [esperado: reclamacao]
    ✓ 'aceita cartao' -> pagamento (18.05%) [esperado: pagamento]
    ✓ 'tchau obrigado' -> despedida (49.24%) [esperado: despedida]

    Acurácia nos testes: 6/6

✓ Q5 OK!


### Questão 6: Classificar com Threshold (12 pontos)

### Objetivo
Classificar texto com threshold de confiança, indicando se a classificação é confiável o suficiente para uso automático.

### Entradas esperadas
- `texto: str` mensagem do usuário.
- `classificador` já treinado.
- `vetorizador` já treinado.
- `threshold: float` valor mínimo de confiança (default `0.5`).

### Processamento obrigatório
Implemente `classificar_com_threshold(texto, classificador, vetorizador, threshold)` de forma que:

1. Use `classificar_intencao()` para obter predição e confiança.
2. Compare confiança com threshold.
3. Retorne dicionário com resultado completo.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `classificar_com_threshold(texto, classificador, vetorizador, threshold=0.5) -> dict`
- O dicionário retornado deve conter **exatamente** as chaves:
  - `'intencao'`: str com a classe predita
  - `'confianca'`: float com a probabilidade
  - `'aceito'`: bool indicando se `confiança >= threshold`

### Restrições técnicas
- O threshold default deve ser `0.5`.
- `aceito` deve ser `True` se e somente se `confiança >= threshold`.
- Use `classificar_intencao()` internamente (não reimplemente a lógica).

### Observações de continuidade
Esta função é usada no Projeto Final para decidir se a resposta automática é confiável ou se deve escalar para atendimento humano. O campo `aceito` determina o fluxo de roteamento.

In [13]:
def classificar_com_threshold(texto: str, classificador, vetorizador, threshold: float = 0.5) -> dict:
    """
    Classifica texto com threshold de confiança.

    Args:
        texto: Mensagem do usuário
        classificador: Classificador treinado
        vetorizador: Vetorizador treinado
        threshold: Confiança mínima (default 0.5)

    Returns:
        dict com 'intencao', 'confianca', 'aceito'
    """
    # === SEU CÓDIGO AQUI ===
    intencao, confianca = classificar_intencao(texto, classificador, vetorizador)

    return {
        'intencao': intencao,
        'confianca': confianca,
        'aceito': confianca >= threshold
    }

In [14]:
# Teste Q6
r1 = classificar_com_threshold("oi bom dia", classificador, vetorizador, 0.3)
r2 = classificar_com_threshold("xyz abc palavras aleatorias", classificador, vetorizador, 0.9)

print("Q6: Classificação com threshold")
print(f"    'oi bom dia' (threshold=0.3): {r1}")
print(f"    'xyz abc...' (threshold=0.9): {r2}")

# Validações obrigatórias
assert isinstance(r1, dict), "Retorno deve ser dict"
assert set(r1.keys()) == {'intencao', 'confianca', 'aceito'}, \
    "Retorno deve conter exatamente as chaves: 'intencao', 'confianca', 'aceito'"

assert isinstance(r1['intencao'], str), "'intencao' deve ser str"
assert isinstance(r1['confianca'], float), "'confianca' deve ser float"
assert isinstance(r1['aceito'], bool), "'aceito' deve ser bool"
assert 0.0 <= r1['confianca'] <= 1.0, "'confianca' deve estar em [0, 1]"

# Verificar lógica do threshold (necessariamente verdadeira)
texto_teste = "oi"
int_base, conf_base = classificar_intencao(texto_teste, classificador, vetorizador)

# Threshold exatamente igual à confiança deve aceitar (>=)
r_eq = classificar_com_threshold(texto_teste, classificador, vetorizador, conf_base)
assert r_eq['aceito'] is True, "Deve aceitar quando confiança == threshold (>=)"

# Threshold um pouco maior que a confiança deve rejeitar
r_gt = classificar_com_threshold(texto_teste, classificador, vetorizador, conf_base + 1e-12)
assert r_gt['aceito'] is False, "Deve rejeitar quando confiança < threshold"

# Verificar consistência com classificar_intencao (efeito)
r_check = classificar_com_threshold(texto_teste, classificador, vetorizador, 0.5)
assert r_check['intencao'] == int_base, "intencao deve ser a mesma retornada por classificar_intencao()"
assert abs(r_check['confianca'] - conf_base) < 1e-12, "confianca deve ser a mesma retornada por classificar_intencao()"

print("\n✓ Q6 OK!")


Q6: Classificação com threshold
    'oi bom dia' (threshold=0.3): {'intencao': 'saudacao', 'confianca': 0.5098075993883413, 'aceito': True}
    'xyz abc...' (threshold=0.9): {'intencao': 'faq', 'confianca': 0.1539512572865749, 'aceito': False}

✓ Q6 OK!


### Questão 7: Classificar em Lote (10 pontos)

### Objetivo
Classificar múltiplos textos de forma eficiente usando vetorização em lote.

### Entradas esperadas
- `textos: list[str]` lista de mensagens a classificar.
- `classificador` já treinado.
- `vetorizador` já treinado.

### Processamento obrigatório
Implemente `classificar_lote(textos, classificador, vetorizador)` de forma que:

1. Vetorize **todos** os textos de uma vez usando `transform(textos)`.
   - Isso é mais eficiente que vetorizar um por um.
2. Obtenha todas as predições usando `predict()`.
3. Obtenha todas as probabilidades usando `predict_proba()`.
4. Para cada texto, extraia a confiança da classe predita.
5. Retorne lista de tuplas `(intenção, confiança)`.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `classificar_lote(textos: list, classificador, vetorizador) -> list`
- A lista retornada deve ter o mesmo tamanho da entrada.
- Cada elemento deve ser tupla `(intenção: str, confiança: float)`.

### Restrições técnicas
- Vetorize todos os textos de uma vez (não use loop para vetorizar).
- A ordem dos resultados deve corresponder à ordem da entrada.
- Use tipos nativos (`str`, `float`) nos retornos.

### Observações de continuidade
Esta função é útil para processamento em lote, análise de logs de conversas e avaliação de desempenho. No Projeto Final, pode ser usada para análise retrospectiva de atendimentos.

In [15]:
def classificar_lote(textos: list, classificador, vetorizador) -> list:
    """
    Classifica lista de textos em lote.

    Args:
        textos: Lista de mensagens
        classificador: Classificador treinado
        vetorizador: Vetorizador treinado

    Returns:
        Lista de tuplas (intenção, confiança)
    """
    # === SEU CÓDIGO AQUI ===
    vetores = vetorizador.transform(textos)

    preds = classificador.predict(vetores)
    probas = classificador.predict_proba(vetores)

    classes = list(classificador.classes_)
    resultados = []

    for i, pred in enumerate(preds):
        idx = classes.index(pred)
        confianca = float(probas[i][idx])
        resultados.append((str(pred), confianca))

    return resultados

In [16]:
# Teste Q7
lote_teste = [
    "oi",
    "onde esta meu pedido",
    "quero devolver",
    "produto quebrado",
    "aceita pix",
    "como funciona",
    "tchau"
]

resultados_lote = classificar_lote(lote_teste, classificador, vetorizador)

print("Q7: Classificação em lote:")
for texto, (intencao, conf) in zip(lote_teste, resultados_lote):
    print(f"    '{texto}' -> {intencao} ({conf:.2%})")

# Validações obrigatórias
assert isinstance(resultados_lote, list), "Retorno deve ser lista"
assert len(resultados_lote) == len(lote_teste), "Deve retornar mesmo número de elementos"
assert all(isinstance(r, tuple) and len(r) == 2 for r in resultados_lote), "Cada item deve ser tupla (intencao, confianca)"
assert all(isinstance(r[0], str) for r in resultados_lote), "Intenção deve ser str"
assert all(isinstance(r[1], float) for r in resultados_lote), "Confiança deve ser float"
assert all(0.0 <= r[1] <= 1.0 for r in resultados_lote), "Confiança deve estar em [0, 1]"

# Ordem preservada + confiança correta (contrato central)
vet = vetorizador.transform(lote_teste)
pred = classificador.predict(vet)
proba = classificador.predict_proba(vet)

# Para cada i, a intenção deve ser pred[i] e a confiança deve ser proba[i][idx(pred[i])]
classes = list(classificador.classes_)
for i, (intencao_i, conf_i) in enumerate(resultados_lote):
    assert intencao_i == str(pred[i]), "A ordem dos resultados deve corresponder à ordem da entrada"
    idx = classes.index(pred[i])
    expected_conf = float(proba[i][idx])
    assert abs(conf_i - expected_conf) < 1e-12, "Confiança deve ser a probabilidade da classe predita"

print("\n✓ Q7 OK!")


Q7: Classificação em lote:
    'oi' -> saudacao (45.43%)
    'onde esta meu pedido' -> rastreamento (54.20%)
    'quero devolver' -> devolucao (38.99%)
    'produto quebrado' -> reclamacao (32.96%)
    'aceita pix' -> pagamento (21.73%)
    'como funciona' -> faq (26.80%)
    'tchau' -> despedida (34.73%)

✓ Q7 OK!


### Questão 8: Avaliar Classificador (13 pontos)

### Objetivo
Calcular métricas de avaliação do classificador no conjunto de teste.

### Entradas esperadas
- `classificador` já treinado.
- `X_test` matriz de features do conjunto de teste.
- `y_test` labels verdadeiros do conjunto de teste.

### Processamento obrigatório
Implemente `avaliar_classificador(classificador, X_test, y_test)` de forma que:

1. Obtenha predições usando `predict()`.
2. Calcule as seguintes métricas usando sklearn.metrics:
   - `accuracy_score`: acurácia geral
   - `precision_score`: precisão (média macro)
   - `recall_score`: recall (média macro)
   - `f1_score`: F1 (média macro)
3. Use `average='macro'` e `zero_division=0` para as métricas.
4. Retorne dicionário com métricas.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `avaliar_classificador(classificador, X_test, y_test) -> dict`
- O dicionário retornado deve conter **exatamente** as chaves:
  - `'accuracy'`: float
  - `'precision'`: float
  - `'recall'`: float
  - `'f1'`: float

### Restrições técnicas
- Use `average='macro'` para precision, recall e f1.
- Use `zero_division=0` para evitar warnings.
- Converta valores para `float` nativo.

### Observações de continuidade
Esta função fornece métricas padronizadas para comparar diferentes configurações do classificador. No Projeto Final, as métricas podem ser exibidas no painel de administração.

In [17]:
def avaliar_classificador(classificador, X_test, y_test) -> dict:
    """
    Avalia classificador com métricas padrão.

    Args:
        classificador: Classificador treinado
        X_test: Features do conjunto de teste
        y_test: Labels verdadeiros

    Returns:
        dict com 'accuracy', 'precision', 'recall', 'f1'
    """
    # === SEU CÓDIGO AQUI ===
    y_pred = classificador.predict(X_test)

    return {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred, average='macro', zero_division=0)),
        'recall': float(recall_score(y_test, y_pred, average='macro', zero_division=0)),
        'f1': float(f1_score(y_test, y_pred, average='macro', zero_division=0))
    }


In [18]:
# Teste Q8
metricas = avaliar_classificador(classificador, X_test, y_test)

print("Q8: Métricas de avaliação:")
for metrica, valor in metricas.items():
    print(f"    {metrica}: {valor:.4f} ({valor:.2%})")


# Validações obrigatórias
assert isinstance(metricas, dict), "Retorno deve ser dict"
assert set(metricas.keys()) == {'accuracy', 'precision', 'recall', 'f1'}, \
    "Retorno deve conter exatamente as chaves: 'accuracy', 'precision', 'recall', 'f1'"

for chave in ['accuracy', 'precision', 'recall', 'f1']:
    assert isinstance(metricas[chave], float), f"'{chave}' deve ser float"
    assert 0.0 <= metricas[chave] <= 1.0, f"'{chave}' deve estar em [0, 1]"


# Verificar cálculo correto (efeito)
y_pred = classificador.predict(X_test)
expected = {
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'precision': float(precision_score(y_test, y_pred, average='macro', zero_division=0)),
    'recall': float(recall_score(y_test, y_pred, average='macro', zero_division=0)),
    'f1': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
}

for k in expected:
    assert abs(metricas[k] - expected[k]) < 1e-12, f"Métrica '{k}' calculada incorretamente"

print("\n✓ Q8 OK!")


Q8: Métricas de avaliação:
    accuracy: 0.7857 (78.57%)
    precision: 0.8350 (83.50%)
    recall: 0.7857 (78.57%)
    f1: 0.7973 (79.73%)

✓ Q8 OK!


---

## Demonstração: Classificador de Intenções

Esta seção valida a integração mínima do sistema implementado.

Requisitos:
- Execute a demonstração para verificar que:
  - o classificador retorna intenções coerentes;
  - o threshold controla corretamente quando aceitar ou rejeitar;
  - as métricas indicam desempenho razoável.
- Se o comportamento observado for inconsistente, trate como defeito de contrato em uma das funções Q1–Q8.

In [19]:
def top2_intencoes(texto: str, classificador, vetorizador):
    """Retorna top-2 (classe, prob) para um único texto."""
    v = vetorizador.transform([texto])
    proba = classificador.predict_proba(v)[0]              # (n_classes,)
    classes = np.array(classificador.classes_, dtype=object)

    idxs = np.argsort(proba)[::-1][:2]                     # top-2
    return [(str(classes[i]), float(proba[i])) for i in idxs]


print("DEMONSTRAÇÃO DO CLASSIFICADOR DE INTENÇÕES")
print("=" * 72)

mensagens_demo = [
    "ola tudo bem",
    "cadê meu pedido PED-123",
    "quero devolver esse produto",
    "produto veio todo quebrado absurdo",
    "aceita cartão de crédito parcelado",
    "como funciona o frete grátis",
    "obrigado pela ajuda tchau",
    "asdfgh jklqwerty",  # texto sem sentido
]

threshold_demo = 0.4

def barra_confianca(p: float, width: int = 20) -> str:
    filled = int(round(p * width))
    filled = max(0, min(width, filled))
    return "█" * filled + "░" * (width - filled)

print(f"\nClassificação com threshold={threshold_demo:.2f}")
print("-" * 72)
print(f"{'STATUS':9s} {'INTENÇÃO':14s} {'CONF':6s} {'BARRA':22s} MENSAGEM")
print("-" * 72)

aceitas = 0
escaladas = 0
confs_aceitas = []
confs_escaladas = []
intencoes = []

for msg in mensagens_demo:
    result = classificar_com_threshold(msg, classificador, vetorizador, threshold_demo)
    top2 = top2_intencoes(msg, classificador, vetorizador)
    (t1, p1), (t2, p2) = top2
    intencao = result['intencao']
    conf = result['confianca']
    aceito = result['aceito']

    status = "ACEITO" if aceito else "ESCALAR"
    icon = "✓" if aceito else "?"
    margem = conf - threshold_demo

    if aceito:
        aceitas += 1
        confs_aceitas.append(conf)
    else:
        escaladas += 1
        confs_escaladas.append(conf)

    intencoes.append(intencao)

    msg_curta = (msg[:45] + "...") if len(msg) > 48 else msg
    bar = barra_confianca(conf)

    print(f"{icon} {status:7s} {intencao:14s} {conf:6.2%} {bar}  {msg_curta}")
    print(f"  {'':9s} {'':14s} {'':6s} {'':22s} (margem vs thr: {margem:+.2%})")
    print(f"  {'':9s} {'':14s} {'':6s} {'':22s} top-2: {t1} ({p1:.2%}), {t2} ({p2:.2%})")


print("-" * 72)

# Resumo de roteamento
import numpy as np
from collections import Counter

dist = Counter(intencoes)
top_int, top_count = dist.most_common(1)[0]

print("RESUMO DE ROTEAMENTO")
print("-" * 72)
print(f"  Aceitas : {aceitas}/{len(mensagens_demo)}")
print(f"  Escalar : {escaladas}/{len(mensagens_demo)}")
print(f"  Intenção mais frequente: {top_int} ({top_count})")

if confs_aceitas:
    print(f"  Confiança média (aceitas): {float(np.mean(confs_aceitas)):.2%}")
if confs_escaladas:
    print(f"  Confiança média (escaladas): {float(np.mean(confs_escaladas)):.2%}")

print("\n" + "=" * 72)
print("MÉTRICAS FINAIS")
print("-" * 72)

metricas_final = avaliar_classificador(classificador, X_test, y_test)
for k in ['accuracy', 'precision', 'recall', 'f1']:
    valor = metricas_final[k]
    bar = "█" * int(round(valor * 20))
    print(f"  {k:12s}: {bar:<20s} {valor:.4f} ({valor:.2%})")

print("=" * 72)


DEMONSTRAÇÃO DO CLASSIFICADOR DE INTENÇÕES

Classificação com threshold=0.40
------------------------------------------------------------------------
STATUS    INTENÇÃO       CONF   BARRA                  MENSAGEM
------------------------------------------------------------------------
? ESCALAR saudacao       32.05% ██████░░░░░░░░░░░░░░  ola tudo bem
                                                         (margem vs thr: -7.95%)
                                                         top-2: saudacao (32.05%), despedida (13.98%)
✓ ACEITO  rastreamento   50.18% ██████████░░░░░░░░░░  cadê meu pedido PED-123
                                                         (margem vs thr: +10.18%)
                                                         top-2: rastreamento (50.18%), devolucao (9.15%)
? ESCALAR devolucao      38.80% ████████░░░░░░░░░░░░  quero devolver esse produto
                                                         (margem vs thr: -1.20%)
                                   

---

## Resumo - Funções para o Projeto Final

As seguintes funções e artefatos serão consumidos diretamente no Projeto Final:

| Função                           | Responsabilidade                | Uso no Projeto Final                         |
| -------------------------------- | ------------------------------- | -------------------------------------------- |
| `texto_para_vetor_tfidf()`       | Vetorização 1 texto (TF-IDF)    | Utilitário interno / múltiplos vetorizadores |
| `preparar_dados_classificacao()` | Setup do vetorizador e split    | Inicialização                                |
| `treinar_classificador()`        | Treinar modelo                  | Setup do chatbot                             |
| `otimizar_hiperparametros()`     | Buscar melhores hiperparâmetros | Ajuste de modelo (opcional)                  |
| `classificar_intencao()`         | **Classificação principal**     | Roteamento de mensagens                      |
| `classificar_com_threshold()`    | Classificação + confiança       | Decisão de escalação                         |
| `classificar_lote()`             | Classificação em lote           | Logs / avaliação offline                     |
| `avaliar_classificador()`        | Métricas de desempenho          | Painel de administração                      |


### Expansões no Projeto Final

No Projeto Final, você irá **expandir** a função `classificar_com_threshold()` para:

1. Escalar para atendimento humano quando confiança estiver abaixo do threshold
2. Registrar métricas de confiança e decisão em sistema de analytics

Requisito de estabilidade:
- Não altere assinaturas base, apenas adicione funcionalidades.

---

## Checklist de Entrega

Antes de submeter, valide objetivamente:

- [ ] Todas as funções estão implementadas e executam sem exceções.
- [ ] O notebook executa do início ao fim, em ordem, sem intervenção manual.
- [ ] Todas as funções preservam:
  - assinatura
  - tipos de retorno
  - parâmetros obrigatórios do classificador
- [ ] A demonstração final executa e retorna saídas compatíveis com a especificação.
- [ ] As métricas de avaliação são calculadas corretamente e retornam valores válidos no intervalo [0, 1].

**Envie via Moodle até a data limite.**
